# VQ-VAE — Neural Discrete Representation Learning (2017)

_Papers · Generative Models_

**Snap each encoder output to its nearest vector in a small learned codebook, so the latent code becomes a grid of discrete symbols.**

---

This notebook is a practice scaffold for the **VQ-VAE — Neural Discrete Representation Learning (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, torchvision.transforms as T

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
BETA = 0.25                                                  # commitment cost (paper: 0.25)

# --- 0. Sanity-check the lesson's worked example (one vector, K=3, D=2). ---
ze0 = torch.tensor([0.9, -0.4])
cb0 = torch.tensor([[1.0, 0.0], [-1.0, 1.0], [0.2, -0.8]])  # 3 codebook rows
d0  = ((ze0[None, :] - cb0) ** 2).sum(1)                     # squared distances (Eqn. 2)
k0  = int(d0.argmin())                                       # nearest-codebook lookup
zq0 = cb0[k0]
codebook0   = ((ze0.detach() - zq0) ** 2).sum().item()      # ||sg[z_e]-e||^2
commitment0 = BETA * ((ze0 - zq0.detach()) ** 2).sum().item()
print(f"worked example: dists={d0.tolist()}  nearest k={k0}  z_q={zq0.tolist()}")
print(f"                codebook term={codebook0:.4f}  commitment={commitment0:.4f}")
# worked example: dists=[0.17, 5.57, 0.65]  nearest k=0  z_q=[1.0, 0.0]
#                 codebook term=0.1700  commitment=0.0425


# --- 1. The nearest-codebook quantizer (the novel module). ---
class VectorQuantizer(nn.Module):
    def __init__(self, K=64, D=16, beta=BETA):
        super().__init__()
        self.K, self.D, self.beta = K, D, beta
        self.codebook = nn.Embedding(K, D)                  # the learned table e in R^{K x D}
        self.codebook.weight.data.uniform_(-1/K, 1/K)

    def forward(self, z_e, use_commit=True):
        # z_e: (B, D, H, W) -> flatten the grid to (B*H*W, D)
        B, D, H, W = z_e.shape
        flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)       # one row per grid cell
        # squared distance to every codebook row, then argmin (Eqn. 2)
        d = (flat.pow(2).sum(1, keepdim=True)
             - 2 * flat @ self.codebook.weight.t()
             + self.codebook.weight.pow(2).sum(1))
        idx = d.argmin(1)                                   # chosen symbol per cell
        z_q = self.codebook(idx).view(B, H, W, D).permute(0, 3, 1, 2)  # gather -> (B,D,H,W)
        # the two squared-distance terms of Eqn. 3
        codebook_loss   = F.mse_loss(z_q, z_e.detach())            # moves the codebook (sg on encoder)
        commitment_loss = F.mse_loss(z_e, z_q.detach())           # moves the encoder (sg on codebook)
        vq_loss = codebook_loss + (self.beta * commitment_loss if use_commit else 0.0)
        # straight-through: decoder sees z_q, gradient flows to z_e
        z_q_st = z_e + (z_q - z_e).detach()
        return z_q_st, vq_loss, idx


# --- 2. Encoder / decoder around the quantizer. ---
class VQVAE(nn.Module):
    def __init__(self, D=16, K=64):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.ReLU(),           # 28 -> 14
            nn.Conv2d(32, 32, 4, 2, 1), nn.ReLU(),          # 14 -> 7
            nn.Conv2d(32, D, 1))                            # -> D channels (z_e grid)
        self.vq = VectorQuantizer(K, D)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(D, 32, 4, 2, 1), nn.ReLU(),  # 7 -> 14
            nn.ConvTranspose2d(32, 32, 4, 2, 1), nn.ReLU(), # 14 -> 28
            nn.Conv2d(32, 1, 1), nn.Sigmoid())

    def forward(self, x, use_commit=True):
        z_e = self.enc(x)
        z_q, vq_loss, idx = self.vq(z_e, use_commit=use_commit)
        x_hat = self.dec(z_q)
        return x_hat, vq_loss, z_e, z_q, idx


# --- 3. MNIST (torchvision preinstalled in Colab). ---
full = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(full, range(12000)),
                                     batch_size=128, shuffle=True)


def train_vqvae(use_commit=True, epochs=5):
    torch.manual_seed(0)
    net = VQVAE().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=2e-3)
    net.train()
    for ep in range(epochs):
        tot = 0.0; n = 0
        for xb, _ in loader:
            xb = xb.to(device)
            x_hat, vq_loss, _, _, _ = net(xb, use_commit=use_commit)
            recon = F.mse_loss(x_hat, xb)                   # reconstruction term (Eqn. 3)
            loss = recon + vq_loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += recon.item() * xb.size(0); n += xb.size(0)
        print(f"  epoch {ep}  recon MSE/img {tot/n:.4f}")
    return net


print("\nTraining VQ-VAE (full loss: reconstruction + codebook + 0.25*commitment):")
vq = train_vqvae(use_commit=True)

# --- 4. Reconstructions + how far encoder sits from its chosen codebook vector. ---
vq.eval()
xb, _ = next(iter(loader)); xb = xb.to(device)
with torch.no_grad():
    x_hat, _, z_e, z_q, idx = vq(xb)
recon_err = F.mse_loss(x_hat, xb).item()
gap = (z_e - z_q).pow(2).mean().item()                      # encoder-to-codebook distance
used = idx.unique().numel()                                 # how many of K=64 symbols are used
print(f"\nWITH commitment:  recon MSE={recon_err:.4f}  enc->codebook gap={gap:.4f}  symbols used={used}/64")

# --- 5. ABLATION: drop the commitment loss; the encoder should drift from the codebook. ---
print("\nTraining ABLATION (no commitment loss):")
vq_nc = train_vqvae(use_commit=False)
vq_nc.eval()
with torch.no_grad():
    x_hat2, _, z_e2, z_q2, idx2 = vq_nc(xb)
recon_err2 = F.mse_loss(x_hat2, xb).item()
gap2 = (z_e2 - z_q2).pow(2).mean().item()
used2 = idx2.unique().numel()
print(f"NO commitment:    recon MSE={recon_err2:.4f}  enc->codebook gap={gap2:.4f}  symbols used={used2}/64")
print("Commitment loss keeps z_e near its chosen codebook vector (small gap); removing it lets the encoder drift.")
# (Exact numbers vary by hardware/seed; this is our small run, not the paper's reported result.)
# To SEE reconstructions in Colab:  import matplotlib.pyplot as plt
#   plt.imshow(x_hat[0,0].cpu(), cmap="gray"); plt.show()

## Visualize the data & results

_Does the commitment loss keep the encoder's outputs close to the codebook? Compare a VQ-VAE trained WITH vs WITHOUT the commitment term._

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision, torchvision.transforms as T

# Reproduce the qualitative effect: the commitment loss keeps z_e near the codebook.
torch.manual_seed(0)
BETA = 0.25
full = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=T.ToTensor())
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(full, range(12000)),
                                     batch_size=128, shuffle=True)

class VQ(nn.Module):
    def __init__(self, K=64, D=16):
        super().__init__(); self.cb = nn.Embedding(K, D); self.cb.weight.data.uniform_(-1/K, 1/K)
    def forward(self, z_e, use_commit):
        B,D,H,W = z_e.shape
        flat = z_e.permute(0,2,3,1).reshape(-1,D)
        d = flat.pow(2).sum(1,keepdim=True) - 2*flat@self.cb.weight.t() + self.cb.weight.pow(2).sum(1)
        idx = d.argmin(1)
        z_q = self.cb(idx).view(B,H,W,D).permute(0,3,1,2)
        loss = F.mse_loss(z_q, z_e.detach()) + (BETA*F.mse_loss(z_e, z_q.detach()) if use_commit else 0.0)
        return z_e + (z_q - z_e).detach(), loss, idx

class Net(nn.Module):
    def __init__(self, D=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(1,32,4,2,1), nn.ReLU(), nn.Conv2d(32,32,4,2,1), nn.ReLU(), nn.Conv2d(32,D,1))
        self.vq  = VQ(64, D)
        self.dec = nn.Sequential(nn.ConvTranspose2d(D,32,4,2,1), nn.ReLU(),
                                 nn.ConvTranspose2d(32,32,4,2,1), nn.ReLU(), nn.Conv2d(32,1,1), nn.Sigmoid())
    def forward(self, x, use_commit):
        z_e = self.enc(x); z_q, loss, idx = self.vq(z_e, use_commit); return self.dec(z_q), loss, z_e, z_q, idx

def run(use_commit):
    torch.manual_seed(0); net = Net(); opt = torch.optim.Adam(net.parameters(), 2e-3)
    for ep in range(5):
        for xb,_ in loader:
            x_hat, vq_loss, _, _, _ = net(xb, use_commit)
            loss = F.mse_loss(x_hat, xb) + vq_loss
            opt.zero_grad(); loss.backward(); opt.step()
    net.eval(); xb,_ = next(iter(loader))
    with torch.no_grad():
        x_hat, _, z_e, z_q, idx = net(xb, use_commit)
    gap = (z_e - z_q).pow(2).mean().item(); recon = F.mse_loss(x_hat, xb).item()
    return gap, recon, idx.unique().numel()

for use_commit in (True, False):
    gap, recon, used = run(use_commit)
    print(("WITH commit " if use_commit else "NO commit   "),
          f"enc->codebook gap={gap:.3f}  recon MSE={recon:.3f}  symbols used={used}/64")
# WITH commit : z_e stays near the codebook (small gap); symbols stay in use.
# NO commit   : z_e drifts (gap blows up); fewer symbols used; reconstruction no better.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Encoder output $z_e=(0.0,\,1.0)$; codebook $e_0=(1.0,\,1.0)$, $e_1=(-0.5,\,0.5)$,
            $e_2=(0.0,\,0.0)$. Which symbol is chosen, and what is $z_q$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Squared distances: to $e_0$: $(0-1)^2+(1-1)^2=1.0$. — _Eqn. 2 uses Euclidean distance; squaring is monotone so it does not change the argmin._
- To $e_1$: $(0+0.5)^2+(1-0.5)^2=0.25+0.25=0.5$. To $e_2$: $0^2+1^2=1.0$. — _Compute all $K=3$ candidates._
- Smallest is $0.5$ at $k=1$. — _Nearest-codebook lookup picks the minimum-distance row._

**Answer:** Symbol $k=1$ is chosen, so $z_q=e_1=(-0.5,\,0.5)$.

</details>

**Problem 2.** Using the chosen $e_1$ from the previous question, compute the codebook loss and the commitment
            loss with $\beta=0.25$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Both terms use $\lVert z_e-e_1\rVert^2=0.5$ (from above). — _The $\operatorname{sg}$ changes which variable receives the gradient, not the numeric value._
- Codebook loss $=\lVert\operatorname{sg}[z_e]-e_1\rVert^2=0.5$. — _Encoder frozen; value is the same distance._
- Commitment loss $=\beta\lVert z_e-\operatorname{sg}[e_1]\rVert^2=0.25\times0.5=0.125$. — _Codebook frozen; weighted by $\beta=0.25$._

**Answer:** Codebook loss $=0.5$; commitment loss $=0.125$; their sum $0.625$ adds to the reconstruction term.

</details>

**Problem 3.** ABLATION. You remove the commitment loss (set its weight to $0$) and retrain. What do you expect
            to happen to the encoder outputs and to reconstruction, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the commitment term's job: pull $z_e(x)$ toward its chosen codebook vector. — _It is the only force keeping the encoder's magnitude in check on the encoder side._
- With it gone, the encoder can grow its outputs without penalty and shift them faster than the codebook can follow. — _The codebook moves only via the codebook loss; if the encoder outruns it, encoder vectors sit far from any codebook vector._
- The snapped $z_q$ then poorly represents $z_e$, and training is less stable. — _Large encoder-to-codebook gaps mean the decoder works from a code that ignores the encoder's fine detail._

**Answer:** Without commitment the encoder outputs drift far from the codebook (their magnitude grows and they flip between symbols), so the quantization error rises and reconstruction quality typically degrades / becomes less stable. The notebook prints both runs to show this — our small run, not the paper's reported number.

</details>